In [ ]:
! git clone https://github.com/bert-nmt/bert-nmt

In [ ]:
import os
os.chdir("/kaggle/working/bert-nmt")


In [ ]:
!pwd

In [ ]:
pip install --editable .

In [ ]:
! mkdir -p data-bin

In [ ]:
!sed -i 's/np.float/float/g' /kaggle/working/bert-nmt/fairseq/data/indexed_dataset.py


In [ ]:
os.chdir("examples/translation/") 
! bash prepare-wmt14en2de.sh
os.chdir("../..") 

In [ ]:
# Binarize the dataset:
! TEXT=examples/translation/wmt17_en_de
! fairseq-preprocess --source-lang en --target-lang de \
  --trainpref examples/translation/wmt17_en_de/train \
  --validpref examples/translation/wmt17_en_de/valid \
  --testpref examples/translation/wmt17_en_de/test \
  --destdir data-bin/wmt17_en_de --thresholdtgt 0 --thresholdsrc 0


In [ ]:
# Train the model:
# If it runs out of memory, try to set --max-tokens 1500 instead
! mkdir -p checkpoints/fconv_wmt_en_de
! fairseq-train data-bin/wmt17_en_de \
  --lr 0.5 --clip-norm 0.1 --dropout 0.2 --max-tokens 4000 \
  --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
  --lr-scheduler fixed --force-anneal 50 \
  --arch fconv_wmt_en_de --save-dir checkpoints/fconv_wmt_en_de

# Generate:
! fairseq-generate data-bin/wmt17_en_de \
  --path checkpoints/fconv_wmt_en_de/checkpoint_best.pt --beam 5 --remove-bpe

In [ ]:
! mkdir -p data-bin
! curl https://dl.fbaipublicfiles.com/fairseq/models/wmt14.v2.en-fr.fconv-py.tar.bz2 | tar xvjf - -C data-bin
! curl https://dl.fbaipublicfiles.com/fairseq/data/wmt14.v2.en-fr.newstest2014.tar.bz2 | tar xvjf - -C data-bin
! fairseq-generate data-bin/wmt14.en-fr.newstest2014  \
  --path data-bin/wmt14.en-fr.fconv-py/model.pt \
  --beam 5 --batch-size 128 --remove-bpe | tee /tmp/gen.out

# Compute BLEU score
! grep ^H /tmp/gen.out | cut -f3- > /tmp/gen.out.sys
! grep ^T /tmp/gen.out | cut -f2- > /tmp/gen.out.ref
! fairseq-score --sys /tmp/gen.out.sys --ref /tmp/gen.out.ref

In [ ]:
! fairseq-generate data-bin/wmt14.en-fr.newstest2014  \
  --path data-bin/wmt14.en-fr.fconv-py/model.pt \
  --beam 5 --batch-size 128 --remove-bpe | tee /tmp/gen.out 

In [ ]:
os.chdir("examples/translation/") 
!sed -i 's|\.\./mosesdecoder|./mosesdecoder|' makedataforbert.sh
! bash makedataforbert.sh "en"
os.chdir("../..") 